In [33]:
import anndata as ad
import scanpy as sc
import numpy as np

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import random_split

In [26]:
DATA_DIR = "/home/ubuntu/data/frangieh"
adata = sc.read_h5ad(f"{DATA_DIR}/rna_qc_filtered.h5ad")

In [ ]:
#Task 1 
#Can a classifier identify which treatment condition a cell came from using its gene expression profile?
#Which genes drive the separation?
#Features are gene expression values, target variable is treatment condition
#Neuronal Network



In [27]:
#we want to work only on hvg
sc.pp.normalize_total(adata)

sc.pp.log1p(adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=100
)

adata = adata[:, adata.var.highly_variable]

In [29]:
#build dataloader
class AnnDataset(Dataset):
    def __init__(self, adata, label_key):
        self.X = adata.X
        self.y = adata.obs[label_key].values

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]

        # Convert sparse row to dense if necessary
        if hasattr(x, "toarray"):
            x = x.toarray().squeeze()

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)

        return x, y

In [30]:
#encodeing of label strings into integers
encoder = LabelEncoder()
adata.obs["perturbation_enc"] = encoder.fit_transform(
    adata.obs["perturbation_2"]
)

/tmp/ipykernel_64370/747796243.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["perturbation_enc"] = encoder.fit_transform(


In [31]:
dataset_full = AnnDataset(
    adata,
    label_key="perturbation_enc"
)

In [35]:
#train test split
train_size = int(0.8 * len(dataset_full))
val_size = len(dataset_full) - train_size

train_dataset, val_dataset = random_split(
    dataset_full,
    [train_size, val_size]
)

In [36]:
#loaders to subsequently introduce data into the model
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

In [37]:
#build the model
#ReLU as activation function
#using 0.25 as dropout
class PerturbationClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(100, 64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.net(x)

In [38]:
#inititate the model
model = PerturbationClassifier()
#loss function
criterion = nn.CrossEntropyLoss()
#adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


In [39]:
num_epochs = 20 # how often will the model loop the algorithm

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for X, y in train_loader: # produces batches (set when the loader was defined 128 cells in the batch 
                                                #100 genes per cell
                                                #128 corresponding perturbation labels)

        optimizer.zero_grad() # So gradients don't accumulate

        outputs = model(X) # Forward pass

        loss = criterion(outputs, y) # Compute loss

        loss.backward() # backpropagation

        optimizer.step() # update weigths

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"loss = {running_loss/len(train_loader):.4f}"
    )

Epoch 1: loss = 0.7156
Epoch 2: loss = 0.6702
Epoch 3: loss = 0.6601
Epoch 4: loss = 0.6546
Epoch 5: loss = 0.6496
Epoch 6: loss = 0.6463
Epoch 7: loss = 0.6457
Epoch 8: loss = 0.6434
Epoch 9: loss = 0.6424
Epoch 10: loss = 0.6413
Epoch 11: loss = 0.6395
Epoch 12: loss = 0.6408
Epoch 13: loss = 0.6385
Epoch 14: loss = 0.6386
Epoch 15: loss = 0.6376
Epoch 16: loss = 0.6376
Epoch 17: loss = 0.6367
Epoch 18: loss = 0.6363
Epoch 19: loss = 0.6359
Epoch 20: loss = 0.6366


In [40]:
model.eval() # model evaluation

correct = 0
total = 0

with torch.no_grad():
    for X, y in val_loader:
        outputs = model(X)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

accuracy = correct / total

print(f"Validation accuracy: {accuracy:.3f}")

Validation accuracy: 0.719
